# 02 — Ideal Quantum Benchmarks

This notebook builds the **ideal benchmark suite** for the Adaptive_QEM_IBM study. It imports the reusable circuit implementations from the `circuits/` package, characterizes each circuit, executes them on an ideal `AerSimulator`, evaluates benchmark-specific success metrics where applicable, and saves the resulting dataset for later noise and hardware experiments.

**Important:** The circuit implementations remain in individual `.py` files; this notebook is only the experimental orchestration and analysis layer.

## Benchmark Suite

| Class | Benchmarks |
|---|---|
| Entanglement | Bell state, GHZ-3, GHZ-4, GHZ-5 |
| Quantum communication | Teleportation, Superdense Coding |
| Quantum algorithms | QFT, Grover, QPE |
| Optimization | QAOA |
| Statistical/control | QRNG |

The suite is intentionally heterogeneous so that later QEM experiments can relate error sensitivity to qubit count, circuit depth, and two-qubit-gate requirements.

In [ ]:
from pathlib import Path
import sys
import json
import math
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data' / 'ideal'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'tables'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Ideal data  :', DATA_DIR)
print('Tables      :', RESULTS_DIR)

## 1. Import the Reusable Benchmark Circuits

The following imports correspond to the planned `circuits/` directory. If an import fails, create or correct the corresponding `.py` file before continuing.

In [ ]:
from circuits.bell import bell_phi_plus
from circuits.ghz import create_ghz
from circuits.teleportation import teleportation
from circuits.superdense import superdense_coding
from circuits.qft import qft
from circuits.grover import grover_2qubit
from circuits.qaoa import qaoa_two_node
from circuits.qpe import qpe
from circuits.qrng import qrng

print('All benchmark modules imported successfully.')

## 2. Build the Benchmark Collection

The parameters are fixed here so that the same benchmark definitions can be reused in the noisy-simulation and IBM-hardware stages.

In [ ]:
benchmarks = {
    'Bell_Phi_Plus': bell_phi_plus(),
    'GHZ_3': create_ghz(3),
    'GHZ_4': create_ghz(4),
    'GHZ_5': create_ghz(5),
    'Teleportation': teleportation(),
    'Superdense_00': superdense_coding('00'),
    'Superdense_01': superdense_coding('01'),
    'Superdense_10': superdense_coding('10'),
    'Superdense_11': superdense_coding('11'),
    'QFT_3': qft(3),
    'QFT_4': qft(4),
    'Grover_2Q': grover_2qubit(),
    'QAOA_2Q': qaoa_two_node(gamma=math.pi / 4, beta=math.pi / 8),
    'QPE': qpe(),
    'QRNG_4': qrng(4),
}

print('Number of benchmark circuits:', len(benchmarks))
print('\n'.join(f'{i:02d}. {name}' for i, name in enumerate(benchmarks, start=1)))

In [ ]:
for name, circuit in benchmarks.items():
    print(f'\n===== {name} =====')
    print(circuit)

## 3. Circuit Characterization

These structural metrics will later be joined with IBM calibration data and QEM performance. The characterization is performed before simulation so that the circuit properties are independent of the observed measurement results.

In [ ]:
def characterize_circuit(qc):
    ops = qc.count_ops()
    return {
        'qubits': qc.num_qubits,
        'clbits': qc.num_clbits,
        'depth': qc.depth(),
        'size': qc.size(),
        'h': ops.get('h', 0),
        'x': ops.get('x', 0),
        'z': ops.get('z', 0),
        'rx': ops.get('rx', 0),
        'rz': ops.get('rz', 0),
        'cx': ops.get('cx', 0),
        'cz': ops.get('cz', 0),
        'cp': ops.get('cp', 0),
        'swap': ops.get('swap', 0),
        'measurements': ops.get('measure', 0),
    }

characterization = []
for name, qc in benchmarks.items():
    row = {'circuit': name}
    row.update(characterize_circuit(qc))
    characterization.append(row)

characterization_df = pd.DataFrame(characterization)
characterization_df

## 4. Ideal Simulation

A fixed shot count and simulator seed are used for reproducibility. The ideal results provide the reference distribution against which noisy and hardware results will later be compared.

In [ ]:
from qiskit_aer import AerSimulator

SHOTS = 4096
SEED_SIMULATOR = 42

simulator = AerSimulator()

ideal_counts = {}
for name, qc in benchmarks.items():
    job = simulator.run(
        qc,
        shots=SHOTS,
        seed_simulator=SEED_SIMULATOR,
    )
    ideal_counts[name] = job.result().get_counts()

print(f'Completed ideal simulation for {len(ideal_counts)} circuits.')

In [ ]:
for name, counts in ideal_counts.items():
    print(f'\n{name}:')
    print(counts)

## 5. Generic Distribution Metrics

For each circuit, the notebook records the number of observed states, the most frequent state, and its probability. Benchmark-specific success functions can be added later where a known target set is defined.

In [ ]:
def distribution_metrics(counts, shots):
    most_likely_state, most_likely_count = max(counts.items(), key=lambda item: item[1])
    return {
        'observed_states': len(counts),
        'most_likely_state': most_likely_state,
        'most_likely_probability': most_likely_count / shots,
        'total_shots': sum(counts.values()),
    }

distribution_rows = []
for name, counts in ideal_counts.items():
    row = {'circuit': name}
    row.update(distribution_metrics(counts, SHOTS))
    distribution_rows.append(row)

distribution_df = pd.DataFrame(distribution_rows)
ideal_results_df = characterization_df.merge(distribution_df, on='circuit', how='left')
ideal_results_df

## 6. Benchmark-Specific Success Metrics

The functions below provide deterministic target-state checks for Bell, GHZ, Grover, superdense coding, and QRNG. Teleportation, QFT, QAOA, and QPE are retained as distribution-based benchmarks and will receive more specialized analysis in later stages.

In [ ]:
def probability_of_states(counts, states, shots=SHOTS):
    return sum(counts.get(state, 0) for state in states) / shots

def expected_success(name, counts):
    if name == 'Bell_Phi_Plus':
        return probability_of_states(counts, ['00', '11'])

    if name.startswith('GHZ_'):
        n = int(name.split('_')[1])
        return probability_of_states(counts, ['0' * n, '1' * n])

    if name.startswith('Superdense_'):
        message = name.split('_')[1]
        return counts.get(message, 0) / SHOTS

    if name == 'Grover_2Q':
        return counts.get('11', 0) / SHOTS

    if name == 'QRNG_4':
        # QRNG has no single correct bit string; ideal uniformity is evaluated later.
        return float('nan')

    return float('nan')

ideal_results_df['benchmark_success_probability'] = [
    expected_success(name, ideal_counts[name])
    for name in ideal_results_df['circuit']
]

ideal_results_df[['circuit', 'qubits', 'depth', 'size', 'cx', 'cz', 'swap', 'benchmark_success_probability']]

## 7. QRNG Uniformity Check

For the 4-qubit QRNG benchmark, the expected ideal distribution contains all 16 four-bit strings with approximately equal probability.

In [ ]:
qrng_counts = ideal_counts['QRNG_4']
all_4bit_states = [format(i, '04b') for i in range(16)]
qrng_probabilities = {
    state: qrng_counts.get(state, 0) / SHOTS
    for state in all_4bit_states
}

print('Expected probability per state:', 1 / 16)
print('Observed probability range    :', min(qrng_probabilities.values()), 'to', max(qrng_probabilities.values()))

pd.DataFrame({
    'state': list(qrng_probabilities.keys()),
    'probability': list(qrng_probabilities.values()),
}).sort_values('state')

## 8. Save Ideal Benchmark Data

Two files are generated: a circuit-characterization table and a combined ideal-results table. Raw measurement counts are stored separately as JSON so that later analyses can reproduce probability calculations without rerunning the simulator.

In [ ]:
characterization_path = DATA_DIR / 'ideal_circuit_characterization.csv'
results_path = DATA_DIR / 'ideal_benchmark_results.csv'
counts_path = DATA_DIR / 'ideal_counts.json'

characterization_df.to_csv(characterization_path, index=False)
ideal_results_df.to_csv(results_path, index=False)
counts_path.write_text(json.dumps(ideal_counts, indent=2), encoding='utf-8')

print('Saved:')
print('-', characterization_path)
print('-', results_path)
print('-', counts_path)

## 9. Final Benchmark Summary

This table is the baseline structural dataset for the subsequent noisy-simulation and IBM hardware experiments.

In [ ]:
summary_columns = [
    'circuit', 'qubits', 'depth', 'size', 'cx', 'cz', 'swap',
    'observed_states', 'most_likely_state',
    'most_likely_probability', 'benchmark_success_probability'
]

ideal_results_df[summary_columns].sort_values('circuit')

## Next Step

After confirming the ideal baseline, proceed to **`03_noise_benchmarks.ipynb`**. That notebook will introduce controlled readout, one-qubit, two-qubit, depolarizing, and thermal-relaxation noise and quantify how circuit structure changes error sensitivity.